# Tweet Sentiment Extraction

## 1. Data Preparation
### 1.1 Import Data

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### 1.2 Data Visualization

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_df=pd.read_csv('/kaggle/input/tweet-sentiment-extraction/train.csv')

train_df[:5]

In [ ]:
test_df=pd.read_csv('/kaggle/input/tweet-sentiment-extraction/test.csv')

test_df[:5]

In [ ]:
print(f'Shape of training data:{train_df.shape}')

print(f'Shape of test data:{test_df.shape}')

### 1.3 Data Cleaning
# Filter out rows where selected_text is not contained in the original tweet text

#### 1.3.1 Check for Missing Values

In [ ]:
print(train_df.isnull().sum())
# Drop rows with missing values
train_df = train_df.dropna()

In [ ]:
print(train_df.isnull().sum())

#### 1.3.2 Validate that the selected text is a substring of the input tweet

In [ ]:
def validate_selected_text(row):
    selected = str(row['selected_text']).lower()
    text = str(row['text']).lower()
    return selected in text

In [ ]:
valid_mask = train_df.apply(validate_selected_text, axis=1)
train_df = train_df[valid_mask]
print(f"Valid data rate: {valid_mask.mean():.2%}")

#### 1.3.3 Inspect special characters in the text

In [ ]:
import re

In [ ]:
def check_specific_char_types(df):
    
    # Emoji / pictographs
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags
                           "]+", flags=re.UNICODE)
    
    # URL
    url_pattern = r'http\S+'
    
    # Hashtags
    hashtag_pattern = r'#\w+'
    
    results = {
        'emoji_count': 0,
        'url_count': 0, 
        'hashtag_count': 0
    }
    
    for text in df['text']:
        text_str = str(text)
        if emoji_pattern.search(text_str):
            results['emoji_count'] += 1
        if re.search(url_pattern, text_str):
            results['url_count'] += 1
        if re.search(hashtag_pattern, text_str):
            results['hashtag_count'] += 1
    
    print("Counts of specific character types:")
    for char_type, count in results.items():
        percentage = (count / len(df)) * 100
        print(f"  {char_type}: {count} occurrences ({percentage:.1f}% of texts)")
    
    return results

URLs are typically noise for sentiment span extraction.
Hashtags may carry sentiment-related information (e.g., #love, #worstday).

In [ ]:
def clean_text(text):
        text = str(text)
        text = re.sub(r'http\S+', '', text)
        text = ' '.join(text.split())
        return text

In [ ]:
# Note: df_processed is used only for inspection and exploratory analysis.
# The main modeling and training steps continue to use train_df.
df_processed = train_df.copy()    
df_processed['text_processed'] = df_processed['text'].apply(clean_text)

In [ ]:
df_processed[:10]

Observation: for neutral tweets, the ground truth often equals the full text, so removing URLs may incorrectly alter the target span.


## 2. Method 1 (Feature-based machine learning baseline)

### 2.1 Tokenization

In [ ]:
import nltk
from nltk.tokenize import TweetTokenizer

In [ ]:
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)

In [ ]:
def tokenize_text(text):
    return tokenizer.tokenize(str(text))

In [ ]:
train_df['tokens'] = train_df['text'].apply(tokenize_text)
train_df['sentiment_tokens'] = train_df['selected_text'].apply(tokenize_text)
train_df[:5]

### 2.2 Build a sentiment lexicon

In [ ]:
from collections import Counter

In [ ]:
def build_sentiment_lexicon_with_tokenization(df):
    
    positive_tokens = []
    negative_tokens = []
    
    positive_selected_tokens = df[df['sentiment'] == 'positive']['sentiment_tokens']
    for tokens_list in positive_selected_tokens:
        positive_tokens.extend(tokens_list)
    
    negative_selected_tokens = df[df['sentiment'] == 'negative']['sentiment_tokens']
    for tokens_list in negative_selected_tokens:
        negative_tokens.extend(tokens_list)
    
    positive_token_freq = Counter(positive_tokens)
    negative_token_freq = Counter(negative_tokens)
    
    token_scores = {}
    all_tokens = set(positive_token_freq.keys()) | set(negative_token_freq.keys())
    
    for token in all_tokens:
        pos_count = positive_token_freq.get(token, 0)
        neg_count = negative_token_freq.get(token, 0)
        total = pos_count + neg_count
        
        if total > 5: 
            # Sentiment score: (positive_count - negative_count) normalized to [-1, 1]
            score = (pos_count - neg_count) / total
            token_scores[token] = score
    
    return token_scores

In [ ]:
sentiment_lexicon = build_sentiment_lexicon_with_tokenization(train_df)
print(sentiment_lexicon['happy'])
print(f"Built a sentiment lexicon with {len(sentiment_lexicon)} tokens")

### 2.3 Training

#### 2.3.1 Prepare training data

In [ ]:
def create_token_features(token, position, token_len, sentiment, sentiment_lexicon):
    # Lexicon-based sentiment score for the token
    sentiment_score = sentiment_lexicon.get(token, 0)
    
    # Normalized token position within the tweet
    normalized_position = position / max(token_len-1, 1) 
    
    # One-hot encoding of the sentiment label
    is_positive = 1 if sentiment == 'positive' else 0
    is_negative = 1 if sentiment == 'negative' else 0  
    is_neutral = 1 if sentiment == 'neutral' else 0
    
    # Combine features into a simple feature vector
    feature_vector = [
        sentiment_score,
        normalized_position, 
        is_positive,
        is_negative,
        is_neutral
    ]
    
    return feature_vector

In [ ]:
def prepare_training_data_with_tokenization(df, sentiment_lexicon):

    features = []
    labels = []
    text_ids = []
    gt_token = []
    
    for idx, row in df.iterrows():

        tokens = row['tokens']  
        label_tokens = row['sentiment_tokens']
        sentiment = row['sentiment']
        
        if sentiment == 'neutral':
            for i, token in enumerate(tokens):
                feature_vector = create_token_features(token, i, len(tokens), sentiment, sentiment_lexicon)
                
                features.append(feature_vector)
                labels.append(1)
                text_ids.append(row['textID'])
                gt_token.append(token)
            continue
        
        for i, token in enumerate(tokens):
            feature_vector = create_token_features(token, i, len(tokens), sentiment, sentiment_lexicon)
            
            label = 1 if token in label_tokens else 0
            
            features.append(feature_vector)
            labels.append(label)
            text_ids.append(row['textID'])
            gt_token.append(token)
    
    return np.array(features), np.array(labels), text_ids, gt_token

In [ ]:
X, y, text_ids, gt_token = prepare_training_data_with_tokenization(train_df, sentiment_lexicon)
print(f"训练数据形状: {X.shape}")

#### 2.3.2 Custom dataset and dataloader

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
print(X[0])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # 80/20 split
train_dataset = TokenDataset(X_train, y_train)
val_dataset = TokenDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
print(train_dataset[0]) 

#### 2.3.3 Model architecture

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class SentimentExtractionModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super(SentimentExtractionModel, self).__init__()
        
        layers = []
        current_size = input_size
        
        for _ in range(num_layers): #3
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            current_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(current_size, 1))
        layers.append(nn.Sigmoid())  # Probability in [0, 1]
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [ ]:
input_size = X.shape[1] #5
model = SentimentExtractionModel(input_size=input_size, hidden_size=128, num_layers=3)
criterion = nn.BCELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

#### 2.3.4 Model training

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for features, labels in val_loader:
                outputs = model(features).squeeze()
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        train_loss = train_loss / len(train_loader)
        val_loss = val_loss / len(val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    
    return train_losses, val_losses

In [ ]:
print("Training the neural network baseline...")
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30)

### 2.4 Quick sanity-check on a few examples

In [ ]:
def jaccard(str1, str2):
    a = set(str1.lower().split())  
    b = set(str2.lower().split())
    c = a.intersection(b)          
    return float(len(c)) / (len(a) + len(b) - len(c))

In [ ]:
def prepare_test_data_with_tokenization(row, sentiment_lexicon):
    features = []
    gt_token = []
    
    tokens = row['tokens']  
    sentiment = row['sentiment']
    text_id = row['textID']
    labels = []
    
    if sentiment == 'neutral':
        for i, token in enumerate(tokens):
            feature_vector = create_token_features(token, i, len(tokens), sentiment, sentiment_lexicon)
            features.append(feature_vector)
            gt_token.append(token)
            labels.append(1)
        return np.array(features), np.array(labels), [text_id] * len(tokens), gt_token
        
    for i, token in enumerate(tokens):
        feature_vector = create_token_features(token, i, len(tokens), sentiment, sentiment_lexicon)
        features.append(feature_vector)
        gt_token.append(token)
        labels.append(1)
    
    return features, labels, [text_id] * len(tokens), gt_token

In [ ]:
for index, row in train_df[:5].iterrows():
    X, labels, test_ids, gt_test_token = prepare_test_data_with_tokenization(row, sentiment_lexicon)
    print(gt_test_token)
    print(labels)
    model.eval()
    X = torch.tensor(X, dtype=torch.float32)
    word_probs = model(X) 
    print(word_probs)
    print("=====================")
    

## 3. Method 2 (BERT-based token classification baseline)

In [ ]:
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

### 3.1 Dataset construction

In [ ]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text'])
        selected_text = str(row['selected_text'])
        sentiment = row['sentiment']
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze()
        labels = self.create_labels(text, selected_text, input_ids)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels,
            'text': text,
            'selected_text': selected_text,
            'sentiment': sentiment
        }
        
    def create_labels(self, text, selected_text, input_ids):

        tokens = self.tokenizer.convert_ids_to_tokens(input_ids)

        selected_start = text.find(selected_text)
        selected_end = selected_start + len(selected_text)
        
        labels = torch.zeros(len(input_ids), dtype=torch.long)
        
        if selected_start == -1:  
            return labels
        
        current_pos = 0
        for i, token in enumerate(tokens):
            if token in ['[CLS]', '[SEP]', '[PAD]']:
                continue
                
            token_text = token.replace('##', '')
            token_start = text.find(token_text, current_pos)
            if token_start == -1:
                continue
            token_end = token_start + len(token_text)
            current_pos = token_end
            
            if token_start >= selected_start and token_end <= selected_end:
                labels[i] = 1  # Mark token as part of the selected span
        
        return labels

In [ ]:
print("Preparing the BERT dataset...")
train_dataset = TweetDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [ ]:
bert_model = BertModel.from_pretrained(model_name)

In [ ]:
class BertForSentimentExtraction(nn.Module):
    def __init__(self, bert_model, num_labels=2, dropout=0.3):
        super(BertForSentimentExtraction, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        sequence_output = outputs.last_hidden_state
        
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)
        
        return logits

In [ ]:
model = BertForSentimentExtraction(bert_model)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
def train_bert_model(model, train_loader, optimizer, criterion, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            
            loss = criterion(
                outputs.view(-1, outputs.size(-1)), 
                labels.view(-1)
            )
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1} completed. Average loss: {avg_loss:.4f}')

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
print("Training the BERT baseline...")
train_bert_model(model, train_loader, optimizer, criterion, epochs=1)

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 1
}

torch.save(checkpoint, '/kaggle/working/bert_model_checkpoint.pth')
print("Checkpoint saved as bert_model_checkpoint.pth")

In [ ]:
def predict_with_bert(text, sentiment, model, tokenizer, max_length=128):

    model.eval()
    
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        predictions = torch.argmax(outputs, dim=-1).squeeze().cpu().numpy()
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().cpu().numpy())
    
    selected_tokens = []
    for i, (token, pred) in enumerate(zip(tokens, predictions)):
        if pred == 1 and token not in ['[CLS]', '[SEP]', '[PAD]']:
            selected_tokens.append(token)
    
    cleaned_text = tokenizer.convert_tokens_to_string(selected_tokens)
    
    if sentiment == 'neutral' and not cleaned_text.strip():
        return text
    
    return cleaned_text

In [ ]:
print("\n=== Testing BERT ===")
for index, row in train_df[:5].iterrows():
    predicted = predict_with_bert(
        row['text'], 
        row['sentiment'], 
        model, 
        tokenizer
    )
    
    print(f"Tweet: {row['text']}")
    print(f"Sentiment: {row['sentiment']}")
    print(f"Ground truth span: {row['selected_text']}")
    print(f"Predicted span: {predicted}")

    jaccard_score = jaccard(row['selected_text'], predicted)
    print(f"Jaccard similarity: {jaccard_score:.4f}")
    print("=" * 50)